# UAV FP — Optimización aerodinámica del flying wing (AeroSandbox + NeuralFoil)

**Importante sobre la arquitectura de este pipeline:** este notebook corre 100% en Python, afuera de Fusion — la API de Fusion NO se puede llamar desde acá (Fusion solo ejecuta scripts Python dentro de su propio proceso, vía el panel *Scripts and Add-Ins*, no desde un kernel de Jupyter externo). Por eso el flujo queda separado en dos partes:

1. **Este notebook** (afuera de Fusion): corre el análisis/optimización aerodinámica las veces que haga falta, con AeroSandbox + NeuralFoil. Al final, imprime/guarda los parámetros del diseño ganador.
2. **`fusion_generar_ala.py`** (adentro de Fusion, vía *Scripts and Add-Ins*): tomás esos números finales, los cargás como User Parameters en Fusion (el script los crea automáticamente la primera vez, después los editás vos a mano en el diálogo de Fusion), y corrés el script **una sola vez** (o cada vez que cambies un parámetro y quieras regenerar la geometría) — sin que eso dependa de cuántas veces corriste este notebook.

Ejecutá las celdas en orden. Podés volver a correr solo la celda de análisis (sin repetir la instalación/imports) las veces que quieras mientras iterás sobre el diseño.

In [ ]:
# Instalar (una sola vez por entorno):
# %pip install aerosandbox

import aerosandbox as asb
import aerosandbox.numpy as np
import json
from pathlib import Path

## 1. Perfil (2D) — chequeo rápido con NeuralFoil

Esto NO es todavía la performance del flying wing. Es solo para comparar perfiles candidatos antes de decidir cuál usar en el ala 3D.

In [ ]:
wing_airfoil = asb.Airfoil("rg15")  # perfil reflex, tipico de flying wings

aero_2d = wing_airfoil.get_aero_from_neuralfoil(
    alpha=np.linspace(-5, 15, 50),
    Re=3e5,
)
print("CL max (2D, seccion aislada):", np.max(aero_2d["CL"]))

## 2. Función de evaluación del flying wing completo (3D, VLM)

Recibe las variables de diseño y devuelve el L/D real del ala completa (con envergadura finita, flecha, torsión). Esta es la función que después alimenta al optimizador.

In [ ]:
def evaluar_diseno(envergadura, cuerda_raiz, cuerda_punta, flecha, torsion_punta, velocidad=15, alpha=3):
    """Devuelve un dict con CL, CD, Cm y L/D de un flying wing dado un set
    de variables de diseno. envergadura/cuerdas/flecha en metros, torsion en grados."""
    ala = asb.Wing(
        symmetric=True,
        xsecs=[
            asb.WingXSec(xyz_le=[0, 0, 0], chord=cuerda_raiz, twist=0, airfoil=wing_airfoil),
            asb.WingXSec(
                xyz_le=[flecha, envergadura / 2, 0],
                chord=cuerda_punta,
                twist=torsion_punta,
                airfoil=wing_airfoil,
            ),
        ],
    )
    avion = asb.Airplane(wings=[ala])
    resultado = asb.VortexLatticeMethod(
        airplane=avion,
        op_point=asb.OperatingPoint(velocity=velocidad, alpha=alpha),
    ).run()
    return {
        "CL": resultado["CL"],
        "CD": resultado["CD"],
        "Cm": resultado["Cm"],
        "LD": resultado["CL"] / resultado["CD"],
    }

## 3. Evaluar un diseño de ejemplo (baseline)

In [ ]:
diseno_base = dict(
    envergadura=2.40,
    cuerda_raiz=0.40,
    cuerda_punta=0.18,
    flecha=0.30,
    torsion_punta=-4,
)

resultado_base = evaluar_diseno(**diseno_base)
print(resultado_base)

## 4. TODO: loop de optimización (algoritmo genético o gradiente)

Acá va el algoritmo que llama `evaluar_diseno(...)` repetidas veces variando `envergadura`, `cuerda_raiz`, `cuerda_punta`, `flecha`, `torsion_punta` para maximizar L/D (con alguna restricción de margen estático si querés replicar el enfoque del paper de Tran et al.). Dos caminos posibles, a decidir:

- **PyGAD / DEAP** (algoritmo genético, más cercano al paper que encontramos)
- **`asb.Opti`** (optimización por gradiente, nativa de AeroSandbox, más rápida pero requiere que la función sea diferenciable)

Por ahora dejamos esto como placeholder — decime cuál de los dos querés y lo armamos en la próxima iteración.

In [ ]:
# mejor_diseno = diseno_base  # <- reemplazar por el resultado real de la optimizacion cuando este lista

## 5. Exportar el diseño final para pasarlo a Fusion

Esto NO lo lee el script de Fusion automáticamente (recordá: no hay forma de conectar este notebook con Fusion en vivo). Es un registro/log de los valores finales — los mismos números después los tenés que poner en el diálogo de *Cambiar Parámetros* de Fusion (o actualizar `DEFAULT_PARAMS` al principio de `fusion_generar_ala.py`).

In [ ]:
mejor_diseno = diseno_base  # reemplazar cuando tengas el resultado de la optimizacion real

salida = {
    "envergadura_mm": mejor_diseno["envergadura"] * 1000,
    "cuerda_raiz_mm": mejor_diseno["cuerda_raiz"] * 1000,
    "cuerda_punta_mm": mejor_diseno["cuerda_punta"] * 1000,
    "flecha_mm": mejor_diseno["flecha"] * 1000,
    "torsion_punta_deg": mejor_diseno["torsion_punta"],
    "resultado": resultado_base,
}

ruta_salida = Path("wing_design_params.json")
ruta_salida.write_text(json.dumps(salida, indent=2))
print(f"Guardado en {ruta_salida.resolve()}")
print(json.dumps(salida, indent=2))